**生成式 AI 使用说明**：本作业中使用生成式 AI 工具时，适用与协作相同的课程政策。和其他协作者一样，每位学生都必须独立写出自己的解答，不能直接依赖交互输出；提交内容中还应注明协作的性质。使用生成式 AI 工具实质性完成作业内容不符合本作业的精神，也会违反 [Honor Code](https://communitystandards.stanford.edu/policies-and-guidance/honor-code)。

In [ ]:
# 将你的 Google Drive 挂载到 Colab VM。
from google.colab import drive
drive.mount('/content/drive')

# TODO：填写 Drive 中保存解压后
# 作业文件夹，例如 'cs231n/assignments/assignment1/'
FOLDERNAME = 'cs231n/assignments/assignment1/'
assert FOLDERNAME is not None, "[!] Enter the foldername."

# 现在已经挂载 Drive，下面确保
# Colab VM 的 Python 解释器可以加载
# 其中的 Python 文件。
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# 将 CIFAR-10 数据集下载到你的 Drive
# 如果它还不存在。
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
!bash get_datasets.sh
%cd /content/drive/My\ Drive/$FOLDERNAME

# Softmax 分类器练习

*请完成并提交这份 worksheet，包括其中的输出以及 worksheet 外部的任何辅助代码。更多细节请参考课程网站上的 [assignments page](http://vision.stanford.edu/teaching/cs231n/assignments.html)。*

在这个练习中，你将：

- 为 Softmax 分类器实现完全向量化的**损失函数**。
- 实现其**解析梯度**的完全向量化表达式。
- 使用数值梯度**检查你的实现**。
- 使用验证集**调优学习率和正则化**强度。
- 使用 **SGD** **优化**损失函数。
- **可视化**最终学到的权重。

In [ ]:
# 运行本 notebook 的初始化代码。
import random
import numpy as np
from cs231n.data_utils import load_CIFAR10
import matplotlib.pyplot as plt

# 这是一点 IPython magic，让 matplotlib 图像直接在 notebook 中显示，
# 而不是在新窗口中显示。
%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # 设置图像的默认大小
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# 另一点 magic，用于让 notebook 重新加载外部 Python 模块；
# 参考 http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

## CIFAR-10 数据加载与预处理

In [ ]:
# 加载原始 CIFAR-10 数据。
cifar10_dir = 'cs231n/datasets/cifar-10-batches-py'

# 清理变量，避免多次加载数据导致内存问题
try:
   del X_train, y_train
   del X_test, y_test
   print('Clear previously loaded data.')
except:
   pass

X_train, y_train, X_test, y_test = load_CIFAR10(cifar10_dir)

# 作为合理性检查，打印训练数据和测试数据的大小。
print('Training data shape: ', X_train.shape)
print('Training labels shape: ', y_train.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

In [ ]:
# 可视化数据集中的一些样本。
# 这里展示每个类别中的若干训练图像。
classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
num_classes = len(classes)
samples_per_class = 7
for y, cls in enumerate(classes):
    idxs = np.flatnonzero(y_train == y)
    idxs = np.random.choice(idxs, samples_per_class, replace=False)
    for i, idx in enumerate(idxs):
        plt_idx = i * num_classes + y + 1
        plt.subplot(samples_per_class, num_classes, plt_idx)
        plt.imshow(X_train[idx].astype('uint8'))
        plt.axis('off')
        if i == 0:
            plt.title(cls)
plt.show()

In [ ]:
# 将数据划分为训练集、验证集和测试集。此外，我们还会
# 从训练数据中取一个较小的开发集（development set）；
# 开发时使用它可以让代码运行得更快。
num_training = 49000
num_validation = 1000
num_test = 1000
num_dev = 500

# 验证集取自原始训练集中的 num_validation 个样本。
mask = range(num_training, num_training + num_validation)
X_val = X_train[mask]
y_val = y_train[mask]

# 训练集使用原始训练集中的前 num_training 个样本。
mask = range(num_training)
X_train = X_train[mask]
y_train = y_train[mask]

# 我们还会构造一个开发集（development set），它是训练集的一个小子集。
# 这样可以在开发时更快地测试代码。
mask = np.random.choice(num_training, num_dev, replace=False)
X_dev = X_train[mask]
y_dev = y_train[mask]

# 我们使用原始测试集中的前 num_test 个样本作为测试数据。
mask = range(num_test)
X_test = X_test[mask]
y_test = y_test[mask]

print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

In [ ]:
# 预处理：把图像数据 reshape 成行向量
X_train = np.reshape(X_train, (X_train.shape[0], -1))
X_val = np.reshape(X_val, (X_val.shape[0], -1))
X_test = np.reshape(X_test, (X_test.shape[0], -1))
X_dev = np.reshape(X_dev, (X_dev.shape[0], -1))

# 作为合理性检查，打印各份数据的形状。
print('Training data shape: ', X_train.shape)
print('Validation data shape: ', X_val.shape)
print('Test data shape: ', X_test.shape)
print('dev data shape: ', X_dev.shape)

In [ ]:
# 预处理：减去均值图像
# 第一步：基于训练数据计算图像均值
mean_image = np.mean(X_train, axis=0)
print(mean_image[:10]) # 打印前几个元素
plt.figure(figsize=(4,4))
plt.imshow(mean_image.reshape((32,32,3)).astype('uint8')) # 可视化均值图像
plt.show()

# 第二步：从训练和测试数据中减去均值图像
X_train -= mean_image
X_val -= mean_image
X_test -= mean_image
X_dev -= mean_image

# 第三步：追加一列全 1 的 bias 维度（也就是 bias trick），
# 这样分类器只需要优化一个权重矩阵 W。
X_train = np.hstack([X_train, np.ones((X_train.shape[0], 1))])
X_val = np.hstack([X_val, np.ones((X_val.shape[0], 1))])
X_test = np.hstack([X_test, np.ones((X_test.shape[0], 1))])
X_dev = np.hstack([X_dev, np.ones((X_dev.shape[0], 1))])

print(X_train.shape, X_val.shape, X_test.shape, X_dev.shape)

## Softmax 分类器

本节的代码都将写在 `cs231n/classifiers/softmax.py` 中。

你可以看到，我们已经预填了 `softmax_loss_naive` 函数，它使用 for 循环来计算 softmax 损失函数。

In [ ]:
# 评估我们提供的朴素损失实现：
from cs231n.classifiers.softmax import softmax_loss_naive
import time

# 生成一个数值很小的随机 Softmax 分类器权重矩阵。
W = np.random.randn(3073, 10) * 0.0001

loss, grad = softmax_loss_naive(W, X_dev, y_dev, 0.000005)
print('loss: %f' % (loss, ))

# 粗略合理性检查：损失应接近 -log(0.1)。
print('loss: %f' % loss)
print('sanity check: %f' % (-np.log(0.1)))

**内联问题 1**

为什么我们预期损失会接近 `-log(0.1)`？请简要解释。

$\color{blue}{\textit 你的回答：}$ *在此填写*

上面函数返回的 `grad` 现在全为零。请推导并实现 softmax 损失函数的梯度，并直接在 `softmax_loss_naive` 函数内部实现。把新代码穿插到现有函数中会比较方便。

为了检查梯度是否实现正确，你可以用数值方法估计损失函数的梯度，并与你计算出的解析梯度进行比较。我们已经提供了完成这件事的代码：

In [ ]:
# 实现梯度后，用下面的代码重新计算梯度，
# 并用我们提供的函数做梯度检查。

# 计算 W 处的损失及其梯度。
loss, grad = softmax_loss_naive(W, X_dev, y_dev, 0.0)

# 在几个随机选择的维度上计算数值梯度，
# 并将它与你解析计算得到的梯度比较；它们应该几乎完全匹配。
from cs231n.gradient_check import grad_check_sparse
f = lambda w: softmax_loss_naive(w, X_dev, y_dev, 0.0)[0]
grad_numerical = grad_check_sparse(f, W, grad)

# 打开正则化后，再做一次梯度检查。
# 你没有忘记正则化项的梯度吧？
loss, grad = softmax_loss_naive(W, X_dev, y_dev, 5e1)
f = lambda w: softmax_loss_naive(w, X_dev, y_dev, 5e1)[0]
grad_numerical = grad_check_sparse(f, W, grad)

**内联问题 2**

虽然 softmax 损失的梯度检查通常很可靠，但对 SVM 损失来说，偶尔会有某个维度在梯度检查中无法完全匹配。这种差异可能由什么造成？这是否值得担心？请给出一个一维的简单例子，说明 SVM 损失的梯度检查可能失败。改变 margin 会如何影响这种情况发生的频率？

注意，样本 $(x_i, y_i)$ 的 SVM 损失定义为：
$$L_i = \sum_{j\ne y_i}\max(0, s_j - s_{y_i} + \Delta)$$
其中 $j$ 遍历除正确类别 $y_i$ 之外的所有类别，$s_j$ 表示第 $j$ 类的分类器分数，$\Delta$ 是标量 margin。更多信息请参考 [this](https://cs231n.github.io/linear-classify/) 页面中的 “Multiclass Support Vector Machine loss”。

*提示：严格来说，SVM 损失函数并不是处处可微。*

$\color{blue}{\textit 你的回答：}$ *在此填写。*

In [ ]:
# 接下来实现 softmax_loss_vectorized；现在只需要计算损失。
# 梯度稍后再实现。
tic = time.time()
loss_naive, grad_naive = softmax_loss_naive(W, X_dev, y_dev, 0.000005)
toc = time.time()
print('Naive loss: %e computed in %fs' % (loss_naive, toc - tic))

from cs231n.classifiers.softmax import softmax_loss_vectorized
tic = time.time()
loss_vectorized, _ = softmax_loss_vectorized(W, X_dev, y_dev, 0.000005)
toc = time.time()
print('Vectorized loss: %e computed in %fs' % (loss_vectorized, toc - tic))

# 两种实现的损失应该匹配，但向量化实现应该快得多。
print('difference: %f' % (loss_naive - loss_vectorized))

In [ ]:
# 完成 softmax_loss_vectorized 的实现，并计算梯度。
# 以向量化方式计算损失函数的梯度。

# 朴素实现和向量化实现应该匹配。
# 向量化版本仍然应该快得多。
tic = time.time()
_, grad_naive = softmax_loss_naive(W, X_dev, y_dev, 0.000005)
toc = time.time()
print('Naive loss and gradient: computed in %fs' % (toc - tic))

tic = time.time()
_, grad_vectorized = softmax_loss_vectorized(W, X_dev, y_dev, 0.000005)
toc = time.time()
print('Vectorized loss and gradient: computed in %fs' % (toc - tic))

# 损失是一个标量，所以很容易比较两个实现计算出的值。
# 损失是标量，两个实现算出的值可以直接比较。梯度是矩阵，
# 对于梯度矩阵，我们使用 Frobenius 范数进行比较。
difference = np.linalg.norm(grad_naive - grad_vectorized, ord='fro')
print('difference: %f' % difference)

### 随机梯度下降

现在我们已经有了损失和梯度的向量化高效表达式，并且梯度与数值梯度匹配。因此，可以开始用 SGD 最小化损失。本部分代码将写在 `cs231n/classifiers/linear_classifier.py` 中。

In [ ]:
# 在 linear_classifier.py 中完成 LinearClassifier.train() 里的 SGD，
# 然后运行下面的代码。
from cs231n.classifiers import Softmax
softmax = Softmax()
tic = time.time()
loss_hist = softmax.train(X_train, y_train, learning_rate=1e-7, reg=2.5e4,
                      num_iters=1500, verbose=True)
toc = time.time()
print('That took %fs' % (toc - tic))

In [ ]:
# 一个有用的调试方法，是画出损失随迭代次数变化的曲线。
# iteration number:
plt.plot(loss_hist)
plt.xlabel('Iteration number')
plt.ylabel('Loss value')
plt.show()

In [ ]:
# 实现 LinearClassifier.predict 函数，并在训练集和验证集上评估性能。
#
# 你应该能得到约 0.34（大于 0.33）的验证集准确率。
y_train_pred = softmax.predict(X_train)
print('training accuracy: %f' % (np.mean(y_train == y_train_pred), ))
y_val_pred = softmax.predict(X_val)
print('validation accuracy: %f' % (np.mean(y_val == y_val_pred), ))

In [ ]:
# 保存训练好的模型，供 autograder 使用。
softmax.save("softmax.npy")

In [ ]:
# 使用验证集调优超参数（正则化强度和学习率）。
# 你应该尝试不同范围的学习率和正则化强度；
# 如果调得比较仔细，应该能在验证集上得到约 0.365（大于 0.36）的准确率。

# 注意：超参数搜索过程中可能会看到 runtime/overflow warning。
# 这可能由极端数值导致，并不是 bug。

# results 是一个字典：键是 (learning_rate, regularization_strength)，
# 值是 (train_accuracy, val_accuracy)。accuracy 就是正确分类样本所占比例。
results = {}
best_val = -1   # 目前为止见到的最高验证集准确率。
best_softmax = None # 达到最高验证集准确率的 Softmax 对象。

################################################################################
# TODO:                                                                        #
# 编写代码，通过在验证集上调参来选择最佳超参数。对于每一组超参数组合，        #
# 在训练集上训练一个 Softmax，然后计算它在训练集和验证集上的准确率，          #
# 并把这些数值存入 results 字典。此外，把最佳验证集准确率存入 best_val，      #
# 把达到该准确率的 Softmax 对象存入 best_softmax。                            #
#                                                                              #
# 提示：开发验证代码时，先使用较小的 num_iters，避免分类器训练耗时太长；       #
# 当你确认验证代码能正常工作后，再用更大的 num_iters 重新运行。               #
################################################################################

# 下面的范围仅作参考；你可以选择修改这些超参数。
learning_rates = [1e-7, 1e-6]
regularization_strengths = [2.5e4, 1e4]



# 打印结果。
for lr, reg in sorted(results):
    train_accuracy, val_accuracy = results[(lr, reg)]
    print('lr %e reg %e train accuracy: %f val accuracy: %f' % (
                lr, reg, train_accuracy, val_accuracy))

print('best validation accuracy achieved during cross-validation: %f' % best_val)

In [ ]:
# 可视化交叉验证结果
import math
import pdb

# pdb.set_trace()

x_scatter = [math.log10(x[0]) for x in results]
y_scatter = [math.log10(x[1]) for x in results]

# 绘制训练准确率
marker_size = 100
colors = [results[x][0] for x in results]
plt.subplot(2, 1, 1)
plt.tight_layout(pad=3)
plt.scatter(x_scatter, y_scatter, marker_size, c=colors, cmap=plt.cm.coolwarm)
plt.colorbar()
plt.xlabel('log learning rate')
plt.ylabel('log regularization strength')
plt.title('CIFAR-10 training accuracy')

# 绘制验证准确率
colors = [results[x][1] for x in results] # marker 的默认大小是 20
plt.subplot(2, 1, 2)
plt.scatter(x_scatter, y_scatter, marker_size, c=colors, cmap=plt.cm.coolwarm)
plt.colorbar()
plt.xlabel('log learning rate')
plt.ylabel('log regularization strength')
plt.title('CIFAR-10 validation accuracy')
plt.show()

In [ ]:
# 在测试集上评估最佳 Softmax
y_test_pred = best_softmax.predict(X_test)
test_accuracy = np.mean(y_test == y_test_pred)
print('Softmax classifier on raw pixels final test set accuracy: %f' % test_accuracy)

In [ ]:
# 保存最佳 softmax 模型
best_softmax.save("best_softmax.npy")

In [ ]:
# 可视化每个类别学到的权重。
# 根据你选择的学习率和正则化强度，这些可视化结果可能好看，也可能不好看。
#
w = best_softmax.W[:-1,:] # 去掉 bias
w = w.reshape(32, 32, 3, 10)
w_min, w_max = np.min(w), np.max(w)
classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
for i in range(10):
    plt.subplot(2, 5, i + 1)

    # 将权重缩放到 0 到 255 之间
    wimg = 255.0 * (w[:, :, :, i].squeeze() - w_min) / (w_max - w_min)
    plt.imshow(wimg.astype('uint8'))
    plt.axis('off')
    plt.title(classes[i])

**内联问题 3**

描述你可视化出来的 Softmax 分类器权重看起来是什么样子，并简要解释为什么会呈现这种形态。

$\color{blue}{\textit 你的回答：}$ *在此填写*

**内联问题 4** - *True or False*

假设总训练损失定义为所有训练样本逐样本损失的总和。是否可能向训练集中加入一个新样本，使 softmax 损失发生变化，但 SVM 损失保持不变？

$\color{blue}{\textit 你的回答：}$


$\color{blue}{\textit 你的解释：}$